# Bab 1 – Konvensi Umum dan Elemen API scikit-learn

## Ringkasan Bab
Bab ini menjelaskan cara berpikir (filosofi desain) di balik scikit-learn dan beberapa aturan API yang dipakai
hampir semua model: **estimator** (`fit`), **predictor** (`predict`), **transformer** (`transform`), **pipeline**,
pencarian **hyperparameter**, atribut umum, serta **tag metadata** pada estimator. Kalau aturan dasar ini sudah dipahami,
kelas scikit-learn apa pun akan terasa familiar karena cara pakainya mirip.

**Isi bab:** Filosofi desain · Estimator · Transformer dan `transform()` · Estimator kustom · Pipeline ·
Pencarian hyperparameter · Atribut dan metode umum · Tag metadata · Praktik terbaik penggunaan API.

In [1]:
import numpy as np, pandas as pd
import sklearn
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.8.0


## 1. Filosofi Desain scikit-learn
scikit-learn dibangun dengan beberapa prinsip (Buitinck dkk., 2013):

| Prinsip | Artinya |
|---|---|
| **Konsisten** | Semua objek memakai metode yang seragam (`fit`, `predict`, `transform`, `score`). |
| **Bisa diperiksa** | Hyperparameter dapat dibaca langsung (`model.C`), sedangkan hasil belajar diberi akhiran garis bawah (`model.coef_`). |
| **Tidak membuat kelas data baru** | Data cukup berupa array NumPy, matriks sparse SciPy, atau DataFrame. |
| **Komposisi** | Alur yang rumit disusun dari blok sederhana (`Pipeline`, `ColumnTransformer`). |
| **Nilai default yang masuk akal** | Model langsung bisa dipakai tanpa pengaturan, dan baru disetel kalau diperlukan. |

In [2]:
X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
X_train.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
130,7.4,2.8,6.1,1.9
122,7.7,2.8,6.7,2.0
81,5.5,2.4,3.7,1.0
71,6.1,2.8,4.0,1.3
89,5.5,2.5,4.0,1.3


## 2. Memahami Estimator
**Estimator** adalah objek apa pun yang belajar dari data, dengan metode `fit(X, y)`.
* **Hyperparameter** ditentukan di `__init__`, yaitu sebelum pelatihan.
* **Parameter hasil belajar** dibuat saat `fit` dan namanya berakhiran `_` (misalnya `coef_`, `classes_`).

Estimator yang juga punya `predict(X)` disebut *predictor*, dan biasanya dilengkapi `predict_proba` dan `score`.

In [3]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(C=1.0, max_iter=200)       # hyperparameters
print("Before fit, has coef_ ?", hasattr(clf, "coef_"))
clf.fit(X_train, y_train)                           # learning
print("After fit,  has coef_ ?", hasattr(clf, "coef_"))
print("Learned classes_:", clf.classes_)
print("coef_ shape:", clf.coef_.shape)
print("Test accuracy:", clf.score(X_test, y_test))

Before fit, has coef_ ? False
After fit,  has coef_ ? True
Learned classes_: [0 1 2]
coef_ shape: (3, 4)
Test accuracy: 0.9473684210526315


## 3. Transformer dan Metode `transform()`
**Transformer** mempelajari suatu pemetaan dari data lewat `fit`, lalu menerapkannya lewat `transform`
(`fit_transform` menjalankan keduanya sekaligus). Contohnya `StandardScaler`, `SimpleImputer`, `PCA`, dan `OneHotEncoder`.

**Aturan penting:** `fit` hanya pada data latih, lalu `transform` pada data latih dan data uji.
Kalau tidak, terjadi *data leakage*, yaitu informasi data uji ikut bocor ke proses pelatihan.

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # learns mean_ and scale_ from TRAIN only
X_test_s  = scaler.transform(X_test)        # re-uses train statistics
print("mean_ :", scaler.mean_.round(3))
print("scale_:", scaler.scale_.round(3))
print("Train mean after scaling ~0:", X_train_s.mean(axis=0).round(6))

mean_ : [5.878 3.066 3.764 1.194]
scale_: [0.849 0.442 1.776 0.767]
Train mean after scaling ~0: [-0. -0. -0. -0.]


## 4. Membuat Estimator Kustom
Kita bisa membuat transformer atau estimator sendiri dengan mewarisi `BaseEstimator` (menyediakan `get_params/set_params`)
dan `TransformerMixin` / `ClassifierMixin` (menyediakan `fit_transform` / `score`). Aturannya:
1. `__init__` hanya menyimpan hyperparameter (tanpa logika dan tanpa data).
2. `fit` harus mengembalikan `self`.
3. Atribut hasil belajar diberi akhiran `_`.

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.utils.validation import check_is_fitted, check_array

class LogClipTransformer(BaseEstimator, TransformerMixin):
    """Clips each feature to the training quantile range, then applies log1p."""
    def __init__(self, q=0.99):
        self.q = q
    def fit(self, X, y=None):
        X = check_array(X)
        self.upper_ = np.quantile(X, self.q, axis=0)     # learned attribute
        return self
    def transform(self, X):
        check_is_fitted(self)
        X = np.minimum(check_array(X), self.upper_)
        return np.log1p(np.clip(X, 0, None))

class MajorityClassifier(ClassifierMixin, BaseEstimator):
    """Baseline: always predicts the most frequent class."""
    def fit(self, X, y):
        vals, counts = np.unique(y, return_counts=True)
        self.classes_, self.majority_ = vals, vals[counts.argmax()]
        return self
    def predict(self, X):
        check_is_fitted(self)
        return np.full(len(X), self.majority_)

print(LogClipTransformer(q=0.9).fit_transform(X_train)[:3].round(3))
print("Baseline accuracy:", MajorityClassifier().fit(X_train, y_train).score(X_test, y_test).__round__(3))

[[2.078 1.335 1.93  1.065]
 [2.078 1.335 1.93  1.099]
 [1.872 1.224 1.548 0.693]]
Baseline accuracy: 0.316


## 5. Pipeline dan Otomatisasi Alur Kerja
`Pipeline` merangkai beberapa transformer dan satu estimator akhir menjadi **satu estimator**. Manfaatnya: kode lebih ringkas,
tidak ada kebocoran data (setiap fold cross-validation melatih ulang semua langkah), dan hanya ada satu objek untuk disimpan
atau di-deploy. `make_pipeline` memberi nama tiap langkah secara otomatis.

In [6]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.svm import SVC

pipe = Pipeline([("scale", StandardScaler()), ("svc", SVC(kernel="rbf"))])
pipe.fit(X_train, y_train)
print("Pipeline accuracy:", pipe.score(X_test, y_test))
print("Steps:", pipe.named_steps.keys())
print(make_pipeline(StandardScaler(), SVC()))

Pipeline accuracy: 0.9473684210526315
Steps: dict_keys(['scale', 'svc'])
Pipeline(steps=[('standardscaler', StandardScaler()), ('svc', SVC())])


## 6. Penyetelan Hyperparameter dengan Metode Pencarian
Hyperparameter tidak dipelajari oleh `fit`, jadi kita mencarinya dengan cross-validation:
* **`GridSearchCV`** mencoba semua kombinasi pada grid.
* **`RandomizedSearchCV`** mencoba `n_iter` kombinasi acak, sehingga lebih hemat waktu kalau ruang pencariannya besar.

Di dalam pipeline, parameter ditulis dengan format `namalangkah__parameter`.

In [7]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import loguniform

grid = GridSearchCV(pipe, {"svc__C": [0.1, 1, 10], "svc__gamma": ["scale", 0.01, 0.1]}, cv=5)
grid.fit(X_train, y_train)
print("Grid best params:", grid.best_params_, "| CV acc:", round(grid.best_score_, 3))

rand = RandomizedSearchCV(pipe, {"svc__C": loguniform(1e-2, 1e2), "svc__gamma": loguniform(1e-3, 1)},
                          n_iter=20, cv=5, random_state=0)
rand.fit(X_train, y_train)
print("Random best params:", rand.best_params_, "| CV acc:", round(rand.best_score_, 3))
print("Test accuracy of best grid model:", grid.score(X_test, y_test))

Grid best params: {'svc__C': 1, 'svc__gamma': 0.1} | CV acc: 0.964


Random best params: {'svc__C': np.float64(0.49501595537331966), 'svc__gamma': np.float64(0.08663279761354557)} | CV acc: 0.956
Test accuracy of best grid model: 0.9210526315789473


## 7. Atribut dan Metode Umum
| Jenis | Nama |
|---|---|
| Metode | `fit`, `predict`, `predict_proba`, `decision_function`, `transform`, `fit_transform`, `score`, `get_params`, `set_params` |
| Atribut | `coef_`, `intercept_`, `classes_`, `feature_importances_`, `n_features_in_`, `feature_names_in_` |

`get_params()` dan `set_params()` membuat setiap objek bisa diatur lewat kode. Inilah yang dipakai oleh grid search.

In [8]:
print(clf.get_params())
clf.set_params(C=0.1)
print("n_features_in_:", clf.n_features_in_, "| feature_names_in_:", clf.feature_names_in_)
from sklearn import set_config
set_config(display="diagram")   # rich HTML pipeline diagram in notebooks
pipe

{'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 200, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': None, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
n_features_in_: 4 | feature_names_in_: ['sepal length (cm)' 'sepal width (cm)' 'petal length (cm)'
 'petal width (cm)']


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scale', ...), ('svc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'


## 8. Tag Metadata
Pada versi scikit-learn yang lebih baru, estimator dapat menyatakan kemampuannya lewat **tag**, misalnya apakah menerima
data sparse, menerima nilai NaN, atau termasuk classifier. Pipeline, meta-estimator, dan `check_estimator` membaca tag ini
untuk menyesuaikan perilakunya. Pada versi terbaru, tag diambil dengan `sklearn.utils.get_tags`.

In [9]:
from sklearn.utils import get_tags
from sklearn.ensemble import HistGradientBoostingClassifier

for est in [LogisticRegression(), HistGradientBoostingClassifier()]:
    t = get_tags(est)
    print(f"{type(est).__name__:32s} estimator_type={t.estimator_type!s:12s} allows_nan={t.input_tags.allow_nan}  sparse={t.input_tags.sparse}")

LogisticRegression               estimator_type=classifier   allows_nan=False  sparse=True
HistGradientBoostingClassifier   estimator_type=classifier   allows_nan=True  sparse=False


## 9. Praktik Terbaik Penggunaan API
1. Selalu bagi data **sebelum** langkah `fit` apa pun, dan bungkus preprocessing beserta model dalam satu `Pipeline`.
2. Atur `random_state` agar hasil bisa diulang, dan gunakan `stratify` saat membagi data klasifikasi.
3. Bandingkan model dengan cross-validation, bukan dari satu pembagian data yang kebetulan bagus.
4. Gunakan DataFrame jika nama fitur penting, dan `set_output(transform="pandas")` agar nama kolom tetap terjaga.
5. Pakai `ColumnTransformer` untuk data dengan tipe kolom campuran.
6. Jangan menyetel model memakai data uji. Simpan data uji untuk evaluasi akhir saja.

In [10]:
from sklearn.model_selection import cross_val_score
Xb, yb = load_breast_cancer(return_X_y=True, as_frame=True)
best_practice = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).set_output(transform="pandas")
scores = cross_val_score(best_practice, Xb, yb, cv=5)
print("CV accuracy: %.3f ± %.3f" % (scores.mean(), scores.std()))

CV accuracy: 0.981 ± 0.007
